# Reviewer revision: canonical server evaluation
Protocol fixed by the author: **5 outer × 4 inner patient folds, 10 repeats**, seed 1729; four tasks, three models and four objectives. Every data-dependent choice stays inside outer training patients.

This notebook prepares the same commands as SERVER_RUN.md. All execution switches default to False. Set switches deliberately on the server; Run All with defaults launches no subprocesses. Do not fit alternative notebook models or select the protocol using outer results. Results and source snapshots from older runs are preserved.


In [ ]:
from pathlib import Path
import json
import os
import shlex
import subprocess
import sys
from IPython.display import display

ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "run_revision.py").is_file()
)
CONFIG = ROOT / "configs/revision.json"
OUTPUT = "results/revision-expanded"
PREFLIGHT = "results/revision-expanded-preflight"
settings = json.loads(CONFIG.read_text())
assert (
    settings["outer_folds"],
    settings["inner_folds"],
    settings["outer_repeats"],
) == (5, 4, 10)
display(settings)

RUN_TESTS = False
RUN_PREFLIGHT = False
RUN_EVALUATION = False
RUN_RESUME = False
RUN_CHECK = False
if RUN_EVALUATION and RUN_RESUME:
    raise ValueError("Choose fresh evaluation or resume, not both.")

env = {
    **os.environ,
    "PYTHON": sys.executable,
    "MPLBACKEND": "Agg",
    "OMP_NUM_THREADS": "1",
    "OPENBLAS_NUM_THREADS": "1",
    "MKL_NUM_THREADS": "1",
}
common = [
    f"PYTHON={sys.executable}",
    f"REVISION_OUTPUT={OUTPUT}",
    f"PREFLIGHT_OUTPUT={PREFLIGHT}",
]


def stage(target, enabled):
    command = ["make", *common, target]
    print(shlex.join(command))
    if enabled:
        folder = ROOT / "logs"
        folder.mkdir(exist_ok=True)
        with (folder / f"notebook-{target}.log").open("w") as stream:
            subprocess.run(
                command,
                cwd=ROOT,
                env=env,
                stdout=stream,
                stderr=subprocess.STDOUT,
                check=True,
            )

## 1. Software checks
Run source ./setup_environment.sh in your server shell first and launch Jupyter from the activated iivv-revision environment. Tests include scientific regressions and synthetic integrations. Synthetic results cannot enter the paper.


In [ ]:
stage("revision-tests", RUN_TESTS)

## 2. Source and partition preflight
Audits source alignment, metadata, quality and test-mean repeatability, and validates all nested patient partitions. It computes audit statistics but fits no classifiers. Inspect mat-to-npz.ipynb before launching the long run.


In [ ]:
stage("revision-preflight", RUN_PREFLIGHT)
path = ROOT / PREFLIGHT / "run_manifest.json"
if path.is_file():
    preflight = json.loads(path.read_text())
    display({key: preflight.get(key) for key in ["status", "input_summary", "error"]})
else:
    print("Preflight has not been run.")

## 3. Full expanded evaluation or identical-protocol resume
Use a persistent server session. This may require millions of inner fits. Fold checkpoints and logs are written independently of notebook output. Do not close a running notebook kernel expecting the computation to finish. A scheduler or terminal run from SERVER_RUN.md is preferable for unattended work.

A fresh evaluation requires a successful preflight with the same configuration and a fresh output directory. Resume only an explicitly interrupted/failed run, with unchanged input hashes and configuration.


In [ ]:
if RUN_EVALUATION:
    preflight = json.loads((ROOT / PREFLIGHT / "run_manifest.json").read_text())
    if preflight.get("status") != "preflight_passed" or preflight.get("synthetic"):
        raise ValueError("Successful real-data preflight required.")
    if {k: preflight["config"][k] for k in settings} != settings:
        raise ValueError(
            "Preflight configuration differs; rerun preflight before evaluation."
        )
stage("revision-evaluation", RUN_EVALUATION)
stage("revision-resume", RUN_RESUME)

## 4. Completion and result handoff
The check verifies patient separation, all expected outer predictions, class identities, matched controls, the prespecified Holm family and input hashes. Review failed candidates and unavailable ablations. Open Figures-and-Tables.ipynb after this succeeds. No cells here import results into manuscripts.


In [ ]:
stage("revision-check", RUN_CHECK)
path = ROOT / OUTPUT / "run_manifest.json"
if path.is_file():
    manifest = json.loads(path.read_text())
    display(
        {
            key: manifest.get(key)
            for key in [
                "status",
                "synthetic",
                "matched_covariate_control_completed",
                "error",
            ]
        }
    )